# Window and step size experiment

Notes:

* Only using 10 of the 30 ADL (smaller dataset plus some actions are very similar)
    * 11: "lift_suitcase_to_floor"
    * 12: "drink_from_glass"
    * 13: "answer_phone"
    * 16: "eat_apple"
    * 20: "use_key_unlock"
    * 21: "pour_water"
    * 23: "brush_teeth"
    * 24: "open_laptop"
    * 28: "open_door"
    * 29: "place_ball_in_basket"


## Imports

In [1]:
%pip install -q numpy pandas scikit-learn matplotlib torch

import os, re
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay, f1_score
from sklearn.model_selection import GroupShuffleSplit, LeaveOneGroupOut

# Run the mvnx_file_reader.ipynb to load MVNX files
%run mvnx_file_reader.ipynb


Note: you may need to restart the kernel to use updated packages.
Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.
Segment labels: ['Pelvis', 'L5', 'L3', 'T12', 'T8', 'Neck', 'Head', 'RightShoulder', 'RightUpperArm', 'RightForeArm', 'RightHand', 'LeftShoulder', 'LeftUpperArm', 'LeftForeArm', 'LeftHand', 'RightUpperLeg', 'RightLowerLeg', 'RightFoot', 'RightToe', 'LeftUpperLeg', 'LeftLowerLeg', 'LeftFoot', 'LeftToe']
Segment IDs: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23]
RightHand positions (first 5 frames): [[0.000326, -0.7332, 1.53858], [0.007659, -0.732989, 1.535369], [0.007659, -0.732989, 1.535369], [5.256316, 1.154359, 0.728047], [5.256684, 1.155001, 0.728414]]
normal
Data types: ['time', 'index', 'tc', 'ms', 'type', 'orientation', 'position', 'velocity', 'acceleration', 'angularVelocity', 'angularAcceleration', 'contacts', 'sensorFreeAcceleration'

### Globals

In [ ]:
# 10 ADL selection
TASK_TO_LABEL = { # U-Limb dataset has 30 total, we're only using 10 of them for this project
    11: "lift_suitcase_to_floor",
    12: "drink_from_glass",
    13: "answer_phone",
    16: "eat_apple",
    20: "use_key_unlock",
    21: "pour_water",
    23: "brush_teeth",
    24: "open_laptop",
    28: "open_door",
    29: "place_ball_in_basket",
}

ALLOWED_TASKS = set(TASK_TO_LABEL.keys())

LEFT_LIMB_SEGS  = ["LeftUpperArm", "LeftForeArm", "LeftHand", "LeftShoulder"]
RIGHT_LIMB_SEGS = ["RightUpperArm", "RightForeArm", "RightHand", "RightShoulder"]

def segments_for_tested_limb(limb: str):
    return LEFT_LIMB_SEGS if limb == "L" else RIGHT_LIMB_SEGS

# You'll have to change the DATA_ROOT variable to point to where you have the data stored on your computer.
# The notebook should be in the same folder as the data, so a relative path should work.
DATA_ROOT = "ULF_in_ADL"
#DATA_ROOT = "D:/UZH_data/ULF_in_ADL"

CHANNELS = ["acceleration", "angularVelocity", "angularAcceleration"]

# Baseline setting used whenever a single dataset is built outside the sweep loop.
WIN_LEN_S = 3.0  # seconds
STEP_S = 0.5     # seconds

# Windowing study sweep settings
SWEEP_WINDOW_LENGTHS = [1.0, 2.0, 3.0, 4.0]
SWEEP_STEP_SIZES = [0.25, 0.5, 1.0, 1.5, 2.0]
MIN_OVERLAP_FRACTION = 0.50

# Cache settings
CACHE_DIR = "cached_datasets"
USE_DATASET_CACHE = True
FORCE_REBUILD_DATASET = False

# Result saving
RESULTS_DIR = "experiment_results"
RESULTS_BASENAME = "windowing_loso_study"

# Training settings
BATCH_SIZE_TRAIN = 128
BATCH_SIZE_EVAL = 256
EPOCHS = 20
LR = 1e-4
WEIGHT_DECAY = 1e-4
DROPOUT = 0.2
KERNEL_SIZE = 5
TCN_CHANNELS = (64, 64, 128, 128)

# Early stopping
EARLY_STOPPING_PATIENCE = 5
EARLY_STOPPING_MIN_DELTA = 0.002
VAL_SIZE = 0.20
RANDOM_STATE = 42

# LOSO evaluation settings
LOSO_MAX_FOLDS = None  # Set to an integer like 4 for a quick pilot run, or keep None for all subjects
PLOT_FIRST_FOLD_CURVES = True
SAVE_PER_FOLD_CSV = True


### File Helpers

Helper functions to find files, parse task from filename, and extract per-segment vectors

In [3]:
# Helper functions to find and parse MVNX files
def iter_mvnx_files(root: str):
    for r, _, files in os.walk(root):
        for fn in files:
            if fn.lower().endswith(".mvnx"):
                yield os.path.join(r, fn)

# Parses participant, task, limb, and rep from filename
def parse_meta_from_filename(filepath: str):
    """
    Expected example: H01_T01_L1.mvnx OR P02_T15_R3.mvnx
    Returns dict: participant, task(int), limb('L'/'R'), rep(int)
    """
    base = os.path.basename(filepath)

    m = re.match(r"^(?P<participant>[A-Za-z]\d{2})_T(?P<task>\d{2})_(?P<limb>[LRlr])(?P<rep>\d)\.mvnx$", base)
    if not m:
        return None

    participant = m.group("participant")
    task = int(m.group("task"))
    limb = m.group("limb")
    rep = int(m.group("rep"))
    return {"participant": participant, "task": task, "limb": limb, "rep": rep}


# Functions to extract frame rate and segment channel data from MVNX dicts
def get_frame_rate_hz(mvnx_dict) -> float:
    # MVNX stores it on subject
    fr = mvnx_dict["mvnx"]["subject"].get("frameRate", None)
    if fr is None:
        raise ValueError("frameRate not found in mvnx['mvnx']['subject']")
    return float(fr)

# Build a map from segment label to its 0-based index in the channel vectors
def get_segment_channel_xyz(mvnx_dict, frame_dict, segment_label: str, channel: str) -> np.ndarray:
    """
    Returns xyz for one segment for a given channel at a given frame.
    Assumes channel in frame is packed as [x,y,z] per segment in segment order.
    """
    seg_map = build_segment_index_map(mvnx_dict)
    if segment_label not in seg_map:
        raise KeyError(f"Segment '{segment_label}' not found in this file.")

    idx = seg_map[segment_label]  # 0-based segment index
    vec = frame_dict.get(channel, None)
    if vec is None:
        raise KeyError(f"Channel '{channel}' not found in this frame.")

    arr = np.asarray(vec, dtype=np.float32).ravel()
    start = 3 * idx
    return arr[start:start+3]

# Only keep frames where type="normal" (some files have "calibration" frames at the start that we want to ignore)
def only_normal_frames(mvnx: dict) -> list[dict]:
    frames = mvnx["mvnx"]["subject"]["frames"]["frame"]
    if isinstance(frames, dict):
        frames = [frames]
    return [fr for fr in frames if str(fr.get("type", "")).lower() == "normal"]

# Generator for start/end indices of sliding windows over N samples
def window_indices(n_samples: int, win_len: int, step: int):
    s = 0
    while s + win_len <= n_samples:
        yield s, s + win_len
        s += step


In [4]:
# quick test
sample_paths = list(iter_mvnx_files(DATA_ROOT))[:5]
print("Found files (sample):")
for p in sample_paths:
    print(" ", os.path.basename(p), parse_meta_from_filename(p))

Found files (sample):
  P17_T02_L3.mvnx {'participant': 'P17', 'task': 2, 'limb': 'L', 'rep': 3}
  P17_T24_L2.mvnx {'participant': 'P17', 'task': 24, 'limb': 'L', 'rep': 2}
  P17_T28_R3.mvnx {'participant': 'P17', 'task': 28, 'limb': 'R', 'rep': 3}
  P17_T26_L1.mvnx {'participant': 'P17', 'task': 26, 'limb': 'L', 'rep': 1}
  P17_T01_L3.mvnx {'participant': 'P17', 'task': 1, 'limb': 'L', 'rep': 3}


Helpers for mvnx frame/channel extractions

In [5]:
# Build windows for one file, returning X_win (num_windows, win_len, num_channels), y_win (num_windows,), g_win (num_windows,)
def windows_file_cnn(filepath: str, win_len_s, step_s):

    meta = parse_meta_from_filename(filepath)
    if meta is None or meta["task"] not in ALLOWED_TASKS:
        return None

    mvnx = load_mvnx(filepath)
    frames = only_normal_frames(mvnx)
    if len(frames) < 10:
        return None

    frame_rate = float(mvnx["mvnx"]["subject"].get("frameRate", 0))
    win_len = max(int(round(win_len_s * frame_rate)), 5)
    step = max(int(round(step_s * frame_rate)), 1)

    segments = segments_for_tested_limb(meta["limb"])
    N = len(frames)

    # Precompute (N,3)
    data = {}
    for seg in segments:
        for ch in CHANNELS:
            arr = np.zeros((N, 3), dtype=np.float32)
            for i, fr in enumerate(frames):
                arr[i] = get_segment_channel_xyz(mvnx, fr, seg, ch)
            data[(ch, seg)] = arr

    X_win, y_win, g_win = [], [], []
    for a, b in window_indices(N, win_len, step):
        cols = []
        for seg in segments:
            for ch in CHANNELS:
                cols.append(data[(ch, seg)][a:b])   # (T,3)
        Xw = np.concatenate(cols, axis=1)          # (T, C)
        X_win.append(Xw)
        y_win.append(TASK_TO_LABEL[meta["task"]])
        g_win.append(meta["participant"])

    if not X_win:
        return None

    return np.stack(X_win, axis=0), np.array(y_win), np.array(g_win)

## Building Dataset
For the initial window/step study, the model uses acceleration, angular velocity, and angular acceleration from the tested-limb segments. Window length and step size are varied while the remaining model configuration is held fixed.


In [6]:
# Dataset caching helpers
def _safe_name(txt: str) -> str:
    txt = str(txt)
    txt = re.sub(r"[^A-Za-z0-9_.-]+", "-", txt)
    return txt.strip("-") or "default"

def make_dataset_cache_path(root: str, channels, win_len_s: float, step_s: float, allowed_tasks=None, cache_dir: str = CACHE_DIR):
    os.makedirs(cache_dir, exist_ok=True)

    root_name = _safe_name(os.path.basename(os.path.normpath(root)) or "dataset")
    channel_name = _safe_name("_".join(channels))
    task_name = "alltasks" if allowed_tasks is None else _safe_name("-".join(map(str, sorted(allowed_tasks))))

    filename = (
        f"dataset_{root_name}"
        f"__ch-{channel_name}"
        f"__win-{win_len_s:g}s"
        f"__step-{step_s:g}s"
        f"__tasks-{task_name}.npz"
    )
    return os.path.join(cache_dir, filename)

# Build the full dataset by iterating over all files and concatenating results
def build_dataset(root: str, win_len_s, step_s):
    X_list, y_list, g_list = [], [], []
    kept = 0
    skipped = 0

    for fp in iter_mvnx_files(root):
        meta = parse_meta_from_filename(fp)
        if meta is None:
            continue
        if meta["task"] not in ALLOWED_TASKS:
            continue

        out = windows_file_cnn(fp, win_len_s=win_len_s, step_s=step_s)
        if out is None:
            skipped += 1
            continue

        Xw, yw, gw = out
        X_list.append(Xw)
        y_list.append(yw)
        g_list.append(gw)
        kept += 1

    if not X_list:
        raise RuntimeError("No windows were built. Check parsing/task filters/DATA_ROOT.")

    X = np.concatenate(X_list, axis=0)   # (N, T, C)
    y = np.concatenate(y_list, axis=0)   # (N,)
    g = np.concatenate(g_list, axis=0)   # (N,)

    print(f"Kept files: {kept} | Skipped files (allowed but failed): {skipped}")
    print("X:", X.shape, "y:", y.shape, "groups:", g.shape)

    return X, y, g

def load_or_build_dataset(root: str, win_len_s, step_s, channels, use_cache=True, force_rebuild=False):
    cache_path = make_dataset_cache_path(
        root=root,
        channels=channels,
        win_len_s=win_len_s,
        step_s=step_s,
        allowed_tasks=ALLOWED_TASKS,
        cache_dir=CACHE_DIR,
    )

    if use_cache and os.path.exists(cache_path) and not force_rebuild:
        print(f"Loading cached dataset from: {cache_path}")
        cached = np.load(cache_path, allow_pickle=True)
        X = cached["X"]
        y = cached["y"]
        g = cached["groups"]
        return X, y, g, cache_path

    print("No cached dataset found. Building dataset from MVNX files...")
    X, y, g = build_dataset(root, win_len_s=win_len_s, step_s=step_s)

    if use_cache:
        np.savez_compressed(
            cache_path,
            X=X.astype(np.float32),
            y=y,
            groups=g,
            channels=np.array(channels, dtype=object),
            win_len_s=np.array([win_len_s], dtype=np.float32),
            step_s=np.array([step_s], dtype=np.float32),
        )
        print(f"Saved dataset cache to: {cache_path}")

    return X, y, g, cache_path

def summarize_dataset(X, y, groups, prefix="Dataset"):
    print(f"{prefix} cache file loaded.")
    print("Dataset shape:", X.shape)
    print("Unique subjects:", len(set(groups)))
    vals, cnts = np.unique(y, return_counts=True)
    print("Class counts:", dict(zip(vals, cnts)))
    print("Example window shape (T,C):", X[0].shape)


### Normalization

Different normalization from SVM

In [7]:
# IMPORTANT:
# Do NOT normalize before LOSO splitting. That would let the held-out subject
# influence scaling. Instead, fit normalization on the training fold only and
# apply it to val/test inside each fold.

def fit_channel_zscore(X: np.ndarray, eps: float = 1e-8):
    """
    Fit one global z-score per feature channel using TRAINING WINDOWS ONLY.

    X: (N, T, C)
    Returns:
        mu: (C,)
        sd: (C,)
    """
    pooled = X.reshape(-1, X.shape[-1]).astype(np.float32)
    mu = pooled.mean(axis=0)
    sd = pooled.std(axis=0)
    sd = np.maximum(sd, eps)
    return mu, sd

def apply_channel_zscore(X: np.ndarray, mu: np.ndarray, sd: np.ndarray) -> np.ndarray:
    """Apply pre-fit channel-wise z-score to (N, T, C)."""
    return ((X.astype(np.float32) - mu) / sd).astype(np.float32)

print("Normalization helpers defined.")
print("Dataset arrays (X, y, groups) are loaded later inside run_windowing_study()")
print("via load_or_build_dataset(...), one window/step configuration at a time.")


Normalization helpers defined.
Dataset arrays (X, y, groups) are loaded later inside run_windowing_study()
via load_or_build_dataset(...), one window/step configuration at a time.


### Note on dataset creation and caching

This notebook does **not** build `X, y, groups` globally near the top.

Instead, for each window/step configuration, `run_windowing_study()` calls:

- `load_or_build_dataset(...)`
- which either loads an existing `.npz` cache or builds the dataset from MVNX files
- then passes `X, y, groups` into `run_loso_experiment(...)`

So if you run a middle cell that tries to inspect `X` before the sweep function has loaded a dataset, you will get a `NameError`.


## Train and Evaluate



Train/test split by subject and encode labels

In [8]:
from sklearn.model_selection import GroupShuffleSplit, LeaveOneGroupOut
import hashlib
from collections import defaultdict
from itertools import product
from datetime import datetime

def encode_labels(y: np.ndarray):
    labels = sorted(list(set(y)))
    lab2i = {lab:i for i, lab in enumerate(labels)}
    i2lab = {i:lab for lab,i in lab2i.items()}
    y_i = np.array([lab2i[v] for v in y], dtype=np.int64)
    return labels, lab2i, i2lab, y_i

def make_grouped_train_val_split(groups_train: np.ndarray, val_size: float = VAL_SIZE, random_state: int = RANDOM_STATE):
    """
    Creates a validation split using training subjects only.
    This is used INSIDE each LOSO fold for early stopping.
    """
    unique_subjects = np.unique(groups_train)

    if len(unique_subjects) < 2:
        raise ValueError("Need at least 2 training subjects to create a validation split.")

    if len(unique_subjects) == 2:
        # Fallback: keep one subject for train and one for validation
        val_subject = unique_subjects[-1]
        val_idx = np.where(groups_train == val_subject)[0]
        train_idx = np.where(groups_train != val_subject)[0]
        return train_idx, val_idx

    gss_val = GroupShuffleSplit(n_splits=1, test_size=val_size, random_state=random_state)
    train_idx, val_idx = next(gss_val.split(np.zeros(len(groups_train)), groups=groups_train))
    return train_idx, val_idx

def hash_array(a: np.ndarray) -> str:
    h = hashlib.md5()
    h.update(a.tobytes())
    return h.hexdigest()

def report_split_integrity(name_a, X_a, g_a, idx_a, name_b, X_b, g_b, idx_b):
    overlap = set(g_a) & set(g_b)
    if overlap:
        print(f"WARNING: Subject(s) in both {name_a} and {name_b}: {overlap}")
    else:
        print(f"No subject overlap detected between {name_a} and {name_b}.")

    hashes_a = {hash_array(X_a[i]): idx_a[i] for i in range(len(X_a))}
    hashes_b = {hash_array(X_b[i]): idx_b[i] for i in range(len(X_b))}
    common = set(hashes_a.keys()) & set(hashes_b.keys())
    print(f"Identical windows in {name_a}/{name_b}: {len(common)}")
    if common:
        for h in list(common)[:5]:
            print(f"Example identical window - {name_a} idx: {hashes_a[h]} | {name_b} idx: {hashes_b[h]}")

def build_loso_splits(X, y_i, groups, max_folds=None):
    logo = LeaveOneGroupOut()
    all_loso_splits = list(logo.split(X, y_i, groups=groups))
    if max_folds is not None:
        all_loso_splits = all_loso_splits[:max_folds]
    return all_loso_splits

def window_overlap_fraction(win_len_s: float, step_s: float) -> float:
    return 1.0 - (step_s / win_len_s)

def is_valid_window_step_combo(win_len_s: float, step_s: float, min_overlap_fraction: float = MIN_OVERLAP_FRACTION) -> bool:
    if step_s <= 0 or win_len_s <= 0:
        return False
    if step_s > win_len_s:
        return False
    return window_overlap_fraction(win_len_s, step_s) >= min_overlap_fraction

def make_windowing_grid(window_lengths=None, step_sizes=None, min_overlap_fraction: float = MIN_OVERLAP_FRACTION):
    window_lengths = SWEEP_WINDOW_LENGTHS if window_lengths is None else window_lengths
    step_sizes = SWEEP_STEP_SIZES if step_sizes is None else step_sizes

    combos = []
    for win_len_s, step_s in product(window_lengths, step_sizes):
        overlap = window_overlap_fraction(win_len_s, step_s)
        if is_valid_window_step_combo(win_len_s, step_s, min_overlap_fraction=min_overlap_fraction):
            combos.append({
                "win_len_s": float(win_len_s),
                "step_s": float(step_s),
                "overlap_fraction": float(overlap),
            })
        else:
            print(f"Skipping invalid combo: win={win_len_s}, step={step_s}, overlap={overlap:.3f}")
    return combos

def ensure_results_dir(results_dir: str = RESULTS_DIR):
    os.makedirs(results_dir, exist_ok=True)
    return results_dir

def make_results_stem(study_name: str, win_len_s: float, step_s: float, results_dir: str = RESULTS_DIR):
    ensure_results_dir(results_dir)
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    study_name = _safe_name(study_name)
    return os.path.join(results_dir, f"{study_name}__win-{win_len_s:g}s__step-{step_s:g}s__{timestamp}")


Simple TCN dataset and model


In [9]:
import copy
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay, f1_score
import matplotlib.pyplot as plt

class WindowDataset(Dataset):
    def __init__(self, X, y_int):
        # Convert (N, T, C) -> (N, C, T) for Conv1d
        self.X = torch.tensor(np.transpose(X, (0, 2, 1)), dtype=torch.float32)
        self.y = torch.tensor(y_int, dtype=torch.long)

    def __len__(self):
        return self.X.shape[0]

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

class Chomp1d(nn.Module):
    """
    Removes the extra right-side padding so the convolution is causal:
    output[t] depends only on x[:t].
    """
    def __init__(self, chomp_size):
        super().__init__()
        self.chomp_size = chomp_size

    def forward(self, x):
        if self.chomp_size == 0:
            return x
        return x[:, :, :-self.chomp_size].contiguous()

class TemporalBlock(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size, dilation, dropout=0.2):
        super().__init__()
        padding = (kernel_size - 1) * dilation  # causal

        self.net = nn.Sequential(
            nn.Conv1d(in_channels, out_channels, kernel_size,
                      padding=padding, dilation=dilation),
            Chomp1d(padding),
            nn.BatchNorm1d(out_channels),
            nn.ReLU(),
            nn.Dropout(dropout),

            nn.Conv1d(out_channels, out_channels, kernel_size,
                      padding=padding, dilation=dilation),
            Chomp1d(padding),
            nn.BatchNorm1d(out_channels),
            nn.ReLU(),
            nn.Dropout(dropout),
        )

        self.downsample = nn.Conv1d(in_channels, out_channels, kernel_size=1) if in_channels != out_channels else None
        self.relu = nn.ReLU()

    def forward(self, x):
        out = self.net(x)
        res = x if self.downsample is None else self.downsample(x)
        return self.relu(out + res)

class SimpleTCN(nn.Module):
    def __init__(self, in_channels, n_classes, channels=TCN_CHANNELS, kernel_size=KERNEL_SIZE, dropout=DROPOUT):
        super().__init__()

        layers = []
        prev_channels = in_channels
        for i, out_channels in enumerate(channels):
            dilation = 2 ** i
            layers.append(
                TemporalBlock(
                    in_channels=prev_channels,
                    out_channels=out_channels,
                    kernel_size=kernel_size,
                    dilation=dilation,
                    dropout=dropout,
                )
            )
            prev_channels = out_channels

        self.tcn = nn.Sequential(*layers)
        self.pool = nn.AdaptiveAvgPool1d(1)
        self.classifier = nn.Linear(prev_channels, n_classes)

    def forward(self, x):
        z = self.tcn(x)
        z = self.pool(z).squeeze(-1)
        return self.classifier(z)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

def make_loaders(X_train, y_train_i, X_val, y_val_i, X_test, y_test_i):
    train_ds = WindowDataset(X_train, y_train_i)
    val_ds   = WindowDataset(X_val, y_val_i)
    test_ds  = WindowDataset(X_test, y_test_i)

    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE_TRAIN, shuffle=True, drop_last=False)
    val_loader   = DataLoader(val_ds, batch_size=BATCH_SIZE_EVAL, shuffle=False, drop_last=False)
    test_loader  = DataLoader(test_ds, batch_size=BATCH_SIZE_EVAL, shuffle=False, drop_last=False)

    return train_ds, val_ds, test_ds, train_loader, val_loader, test_loader

def build_model_and_training_objects(in_channels: int, y_train_i: np.ndarray):
    model = SimpleTCN(
        in_channels=in_channels,
        n_classes=len(labels),
    ).to(device)

    counts = np.bincount(y_train_i, minlength=len(labels)).astype(np.float32)
    w = counts.sum() / (counts + 1e-6)
    w = w / w.mean()
    criterion = nn.CrossEntropyLoss(weight=torch.tensor(w, dtype=torch.float32).to(device))
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    return model, criterion, optimizer

def eval_model(model, loader):
    model.eval()
    ys, ps = [], []
    with torch.no_grad():
        for xb, yb in loader:
            xb = xb.to(device)
            logits = model(xb)
            pred = torch.argmax(logits, dim=1).cpu().numpy()
            ps.append(pred)
            ys.append(yb.numpy())
    y_true = np.concatenate(ys)
    y_pred = np.concatenate(ps)
    return y_true, y_pred

def run_eval_loss_acc(model, loader, criterion):
    model.eval()
    total_loss = 0.0
    ys, ps = [], []
    with torch.no_grad():
        for xb, yb in loader:
            xb = xb.to(device)
            yb = yb.to(device)
            logits = model(xb)
            loss = criterion(logits, yb)
            total_loss += loss.item() * xb.size(0)

            pred = torch.argmax(logits, dim=1)
            ys.append(yb.cpu().numpy())
            ps.append(pred.cpu().numpy())

    y_true = np.concatenate(ys)
    y_pred = np.concatenate(ps)
    avg_loss = total_loss / len(loader.dataset)
    acc = accuracy_score(y_true, y_pred)
    return avg_loss, acc

print("TCN classes and helper functions are ready.")


Using device: cuda
TCN classes and helper functions are ready.


### Train and evaluate

In [10]:
from sklearn.metrics import f1_score

def run_loso_experiment(
    X,
    y,
    groups,
    study_name: str,
    win_len_s: float,
    step_s: float,
    plot_first_fold_curves: bool = PLOT_FIRST_FOLD_CURVES,
    loso_max_folds = LOSO_MAX_FOLDS,
    save_per_fold_csv: bool = SAVE_PER_FOLD_CSV,
    results_dir: str = RESULTS_DIR,
):
    labels, lab2i, i2lab, y_i = encode_labels(y)
    all_loso_splits = build_loso_splits(X, y_i, groups, max_folds=loso_max_folds)

    print(f"Total LOSO folds to run: {len(all_loso_splits)}")
    print("Classes:", labels)
    print("Subjects:", sorted(np.unique(groups)))

    fold_rows = []
    all_y_true = []
    all_y_pred = []
    first_fold_history = None

    for fold_idx, (trainval_idx, test_idx) in enumerate(all_loso_splits, start=1):
        held_out_subject = np.unique(groups[test_idx])
        if len(held_out_subject) != 1:
            raise RuntimeError("LOSO fold should contain exactly one held-out subject.")
        held_out_subject = held_out_subject[0]

        print("=" * 90)
        print(f"LOSO fold {fold_idx}/{len(all_loso_splits)} | test subject: {held_out_subject}")

        # Outer split: one subject held out for test
        X_trainval_raw, X_test_raw = X[trainval_idx], X[test_idx]
        y_trainval_i, y_test_i = y_i[trainval_idx], y_i[test_idx]
        g_trainval, g_test = groups[trainval_idx], groups[test_idx]

        # Inner grouped split for validation (training subjects only)
        train_sub_idx, val_sub_idx = make_grouped_train_val_split(g_trainval, val_size=VAL_SIZE, random_state=RANDOM_STATE)

        X_train_raw, X_val_raw = X_trainval_raw[train_sub_idx], X_trainval_raw[val_sub_idx]
        y_train_i, y_val_i = y_trainval_i[train_sub_idx], y_trainval_i[val_sub_idx]
        g_train, g_val = g_trainval[train_sub_idx], g_trainval[val_sub_idx]

        print(f"Train subjects: {sorted(np.unique(g_train))}")
        print(f"Val subjects:   {sorted(np.unique(g_val))}")
        print(f"Test subject:   {sorted(np.unique(g_test))}")

        # Fit normalization on TRAIN ONLY, then apply to val/test
        mu, sd = fit_channel_zscore(X_train_raw)
        X_train = apply_channel_zscore(X_train_raw, mu, sd)
        X_val   = apply_channel_zscore(X_val_raw, mu, sd)
        X_test  = apply_channel_zscore(X_test_raw, mu, sd)

        # Leakage checks
        report_split_integrity("train", X_train, g_train, trainval_idx[train_sub_idx],
                            "val", X_val, g_val, trainval_idx[val_sub_idx])
        report_split_integrity("train", X_train, g_train, trainval_idx[train_sub_idx],
                            "test", X_test, g_test, test_idx)
        report_split_integrity("val", X_val, g_val, trainval_idx[val_sub_idx],
                            "test", X_test, g_test, test_idx)

        train_ds, val_ds, test_ds, train_loader, val_loader, test_loader = make_loaders(
            X_train, y_train_i, X_val, y_val_i, X_test, y_test_i
        )

        model, criterion, optimizer = build_model_and_training_objects(
            in_channels=train_ds.X.shape[1],
            y_train_i=y_train_i,
        )

        train_losses, val_losses, test_losses = [], [], []
        train_accs, val_accs, test_accs = [], [], []

        epochs_ran = 0
        best_val_acc = -1.0
        best_state = None
        patience_counter = 0

        for epoch in range(1, EPOCHS + 1):
            epochs_ran = epoch
            model.train()
            total_loss = 0.0
            ys, ps = [], []

            for xb, yb in train_loader:
                xb = xb.to(device)
                yb = yb.to(device)

                optimizer.zero_grad()
                logits = model(xb)
                loss = criterion(logits, yb)
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()

                total_loss += loss.item() * xb.size(0)
                pred = torch.argmax(logits, dim=1)
                ys.append(yb.cpu().numpy())
                ps.append(pred.detach().cpu().numpy())

            train_loss = total_loss / len(train_loader.dataset)
            train_acc = accuracy_score(np.concatenate(ys), np.concatenate(ps))

            val_loss, val_acc = run_eval_loss_acc(model, val_loader, criterion)
            test_loss, test_acc = run_eval_loss_acc(model, test_loader, criterion)

            train_losses.append(train_loss)
            val_losses.append(val_loss)
            test_losses.append(test_loss)

            train_accs.append(train_acc)
            val_accs.append(val_acc)
            test_accs.append(test_acc)

            print(
                f"Epoch {epoch:02d} | "
                f"train_loss={train_loss:.4f} train_acc={train_acc:.4f} | "
                f"val_loss={val_loss:.4f} val_acc={val_acc:.4f} | "
                f"test_loss={test_loss:.4f} test_acc={test_acc:.4f}"
            )

            improvement = val_acc - best_val_acc
            if improvement > EARLY_STOPPING_MIN_DELTA:
                best_val_acc = val_acc
                best_state = copy.deepcopy(model.state_dict())
                patience_counter = 0
            else:
                patience_counter += 1
                print(
                    f"  No validation improvement greater than {EARLY_STOPPING_MIN_DELTA:.4f}. "
                    f"Patience: {patience_counter}/{EARLY_STOPPING_PATIENCE}"
                )
                if patience_counter >= EARLY_STOPPING_PATIENCE:
                    print(f"Early stopping triggered at epoch {epoch}.")
                    break

        if best_state is not None:
            model.load_state_dict(best_state)

        y_true_fold, y_pred_fold = eval_model(model, test_loader)

        fold_acc = accuracy_score(y_true_fold, y_pred_fold)
        fold_macro_f1 = f1_score(y_true_fold, y_pred_fold, average="macro", zero_division=0)

        fold_rows.append({
            "fold": fold_idx,
            "test_subject": held_out_subject,
            "win_len_s": win_len_s,
            "step_s": step_s,
            "overlap_fraction": window_overlap_fraction(win_len_s, step_s),
            "n_train_windows": len(train_ds),
            "n_val_windows": len(val_ds),
            "n_test_windows": len(test_ds),
            "n_train_subjects": len(np.unique(g_train)),
            "n_val_subjects": len(np.unique(g_val)),
            "epochs_ran": epochs_ran,
            "best_val_acc": best_val_acc,
            "test_acc": fold_acc,
            "test_macro_f1": fold_macro_f1,
        })

        all_y_true.append(y_true_fold)
        all_y_pred.append(y_pred_fold)

        print(f"Fold {fold_idx} complete | subject={held_out_subject} | "
            f"test_acc={fold_acc:.4f} | test_macro_f1={fold_macro_f1:.4f}")

        if first_fold_history is None:
            first_fold_history = {
                "epochs_ran": epochs_ran,
                "train_losses": train_losses.copy(),
                "val_losses": val_losses.copy(),
                "test_losses": test_losses.copy(),
                "train_accs": train_accs.copy(),
                "val_accs": val_accs.copy(),
                "test_accs": test_accs.copy(),
                "subject": held_out_subject,
            }

    results_df = pd.DataFrame(fold_rows)
    display(results_df)

    all_y_true = np.concatenate(all_y_true)
    all_y_pred = np.concatenate(all_y_pred)

    overall_acc = accuracy_score(all_y_true, all_y_pred)
    overall_macro_f1 = f1_score(all_y_true, all_y_pred, average="macro", zero_division=0)
    mean_fold_acc = results_df["test_acc"].mean()
    std_fold_acc = results_df["test_acc"].std(ddof=1) if len(results_df) > 1 else 0.0
    mean_fold_macro_f1 = results_df["test_macro_f1"].mean()
    std_fold_macro_f1 = results_df["test_macro_f1"].std(ddof=1) if len(results_df) > 1 else 0.0
    mean_epochs_ran = results_df["epochs_ran"].mean()
    mean_best_val_acc = results_df["best_val_acc"].mean()

    print("\n" + "=" * 90)
    print("LOSO SUMMARY")
    print(f"Overall window-level accuracy: {overall_acc:.4f}")
    print(f"Overall window-level macro F1: {overall_macro_f1:.4f}")
    print(f"Mean fold accuracy: {mean_fold_acc:.4f} ± {std_fold_acc:.4f}")
    print(f"Mean fold macro F1: {mean_fold_macro_f1:.4f} ± {std_fold_macro_f1:.4f}")

    print("\nPer-fold results:")
    display(results_df.sort_values("test_acc", ascending=False))

    print("\nClassification report across all LOSO test folds:")
    print(classification_report(all_y_true, all_y_pred, target_names=labels, digits=3, zero_division=0))

    cm = confusion_matrix(all_y_true, all_y_pred, labels=list(range(len(labels))))
    disp = ConfusionMatrixDisplay(cm, display_labels=labels)
    fig, ax = plt.subplots(figsize=(9, 9))
    disp.plot(ax=ax, xticks_rotation=45)
    plt.title(f"LOSO Confusion Matrix | win={win_len_s:g}s step={step_s:g}s")
    plt.show()

    if plot_first_fold_curves and first_fold_history is not None:
        epochs = range(1, first_fold_history["epochs_ran"] + 1)

        plt.figure(figsize=(7, 5))
        plt.plot(epochs, first_fold_history["train_losses"], label="Train loss")
        plt.plot(epochs, first_fold_history["val_losses"], label="Val loss")
        plt.plot(epochs, first_fold_history["test_losses"], label="Test loss")
        plt.xlabel("Epoch")
        plt.ylabel("Loss")
        plt.title(f"TCN Loss vs Epoch | First LOSO fold ({first_fold_history['subject']})")
        plt.legend()
        plt.grid(True)
        plt.show()

        plt.figure(figsize=(7, 5))
        plt.plot(epochs, first_fold_history["train_accs"], label="Train acc")
        plt.plot(epochs, first_fold_history["val_accs"], label="Val acc")
        plt.plot(epochs, first_fold_history["test_accs"], label="Test acc")
        plt.xlabel("Epoch")
        plt.ylabel("Accuracy")
        plt.title(f"TCN Accuracy vs Epoch | First LOSO fold ({first_fold_history['subject']})")
        plt.legend()
        plt.grid(True)
        plt.show()

    results_stem = make_results_stem(study_name=study_name, win_len_s=win_len_s, step_s=step_s, results_dir=results_dir)
    summary_row = pd.DataFrame([{
        "study_name": study_name,
        "win_len_s": win_len_s,
        "step_s": step_s,
        "overlap_fraction": window_overlap_fraction(win_len_s, step_s),
        "n_total_windows": len(y),
        "n_subjects": len(np.unique(groups)),
        "n_classes": len(labels),
        "cache_path": make_dataset_cache_path(DATA_ROOT, CHANNELS, win_len_s, step_s, ALLOWED_TASKS, CACHE_DIR),
        "loso_folds_ran": len(results_df),
        "overall_acc": overall_acc,
        "overall_macro_f1": overall_macro_f1,
        "mean_fold_acc": mean_fold_acc,
        "std_fold_acc": std_fold_acc,
        "mean_fold_macro_f1": mean_fold_macro_f1,
        "std_fold_macro_f1": std_fold_macro_f1,
        "mean_epochs_ran": mean_epochs_ran,
        "mean_best_val_acc": mean_best_val_acc,
    }])

    summary_csv = results_stem + "__summary.csv"
    folds_csv = results_stem + "__folds.csv"
    summary_row.to_csv(summary_csv, index=False)
    if save_per_fold_csv:
        results_df.to_csv(folds_csv, index=False)

    print(f"Saved summary CSV: {summary_csv}")
    if save_per_fold_csv:
        print(f"Saved per-fold CSV: {folds_csv}")

    return {
        "summary_row": summary_row,
        "fold_results": results_df,
        "all_y_true": all_y_true,
        "all_y_pred": all_y_pred,
        "summary_csv": summary_csv,
        "folds_csv": folds_csv if save_per_fold_csv else None,
    }

def run_windowing_study(
    window_lengths = None,
    step_sizes = None,
    min_overlap_fraction: float = MIN_OVERLAP_FRACTION,
    study_name: str = RESULTS_BASENAME,
    use_dataset_cache: bool = USE_DATASET_CACHE,
    force_rebuild_dataset: bool = FORCE_REBUILD_DATASET,
    loso_max_folds = LOSO_MAX_FOLDS,
):
    combos = make_windowing_grid(window_lengths, step_sizes, min_overlap_fraction=min_overlap_fraction)
    if not combos:
        raise RuntimeError("No valid window/step combinations to run.")

    summary_rows = []
    detailed_outputs = []

    for combo_idx, combo in enumerate(combos, start=1):
        win_len_s = combo["win_len_s"]
        step_s = combo["step_s"]
        overlap_fraction = combo["overlap_fraction"]

        print("\n" + "#" * 100)
        print(
            f"WINDOWING RUN {combo_idx}/{len(combos)} | "
            f"win={win_len_s:g}s | step={step_s:g}s | overlap={overlap_fraction:.2%}"
        )

        global WIN_LEN_S, STEP_S
        WIN_LEN_S = win_len_s
        STEP_S = step_s

        X, y, groups, dataset_cache_path = load_or_build_dataset(
            DATA_ROOT,
            win_len_s=win_len_s,
            step_s=step_s,
            channels=CHANNELS,
            use_cache=use_dataset_cache,
            force_rebuild=force_rebuild_dataset,
        )
        summarize_dataset(X, y, groups, prefix=f"Run {combo_idx}")

        run_out = run_loso_experiment(
            X=X,
            y=y,
            groups=groups,
            study_name=study_name,
            win_len_s=win_len_s,
            step_s=step_s,
            plot_first_fold_curves=(PLOT_FIRST_FOLD_CURVES and combo_idx == 1),
            loso_max_folds=loso_max_folds,
            save_per_fold_csv=SAVE_PER_FOLD_CSV,
            results_dir=RESULTS_DIR,
        )

        summary_row = run_out["summary_row"].copy()
        summary_row["dataset_cache_path"] = dataset_cache_path
        summary_rows.append(summary_row)
        detailed_outputs.append(run_out)

    study_df = pd.concat(summary_rows, ignore_index=True)
    study_df = study_df.sort_values(["overall_macro_f1", "overall_acc"], ascending=False).reset_index(drop=True)

    ensure_results_dir(RESULTS_DIR)
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    study_csv = os.path.join(RESULTS_DIR, f"{_safe_name(study_name)}__ALL_RUNS__{timestamp}.csv")
    study_df.to_csv(study_csv, index=False)

    print("\n" + "=" * 100)
    print("WINDOWING STUDY COMPLETE")
    display(study_df)
    print(f"Saved overall study CSV: {study_csv}")

    return study_df, detailed_outputs, study_csv

# Run the full windowing study
windowing_results_df, windowing_outputs, windowing_study_csv = run_windowing_study()

print("\nFinal ranked windowing results:")
display(windowing_results_df)
print(f"Combined study CSV: {windowing_study_csv}")



####################################################################################################
WINDOWING RUN 1/1 | win=4s | step=0.5s | overlap=87.50%
No cached dataset found. Building dataset from MVNX files...
Kept files: 1420 | Skipped files (allowed but failed): 88
X: (14521, 240, 132) y: (14521,) groups: (14521,)
Saved dataset cache to: cached_datasets/dataset_ULF_in_ADL__ch-orientation_position_velocity_acceleration_angularVelocity_angularAcceleration_sensorFreeAcceleration_sensorMagneticField_sensorOrientation_jointAngle_jointAngleXZY__win-4s__step-0.5s__tasks-11-12-13-16-20-21-23-24-28-29.npz
Run 1 cache file loaded.
Dataset shape: (14521, 240, 132)
Unique subjects: 25
Class counts: {np.str_('answer_phone'): np.int64(883), np.str_('brush_teeth'): np.int64(1313), np.str_('drink_from_glass'): np.int64(1148), np.str_('eat_apple'): np.int64(580), np.str_('lift_suitcase_to_floor'): np.int64(917), np.str_('open_door'): np.int64(485), np.str_('open_laptop'): np.int64(2781), np.

AcceleratorError: CUDA error: out of memory
Search for `cudaErrorMemoryAllocation' in https://docs.nvidia.com/cuda/cuda-runtime-api/group__CUDART__TYPES.html for more information.
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.
